# 02 — Baseline vs XGBoost Comparison

**Goal**: Compare the naive prior-leg baseline against our engineered XGBoost model using a **like-for-like held-out evaluation**.

Both models are evaluated on **the same chronological held-out test set** (the final 15% of journeys sorted by `journey_date`). The baseline is never trained on this data; XGBoost is trained strictly on the first 70%. This ensures the comparison is valid: both models face the same unseen future data.

Previous versions of this notebook used TimeSeriesSplit for XGBoost but the full dataset for the baseline — an invalid comparison. This version corrects that.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from src.features.engineering import engineer_all_features
from src.models.xgboost_model import build_model
from src.calibration.conformal import train_and_calibrate

FEATURES = [
    'scheduled_travel_hours', 'distance_km', 'zone_congestion_index',
    'monsoon_flag', 'fog_risk', 'coach_count', 'loco_age_years',
    'prior_leg_delay', 'schedule_buffer_hours', 'day_of_week', 'month', 'is_weekend'
]

## 1. Load & Engineer Data

In [ ]:
df = pd.read_parquet('../data/processed/kaggle_competition_cleaned.parquet')
df = engineer_all_features(df)
print(f"Data shape after feature engineering: {df.shape}")

## 2. Evaluate Naive Baseline

The baseline carries forward the prior observed delay for the same train number. This is a journey-level proxy, not verified physical-rake linkage.

In [ ]:
df_sorted = df.sort_values('journey_date').reset_index(drop=True)
train_end = int(len(df_sorted) * 0.70)
cal_end = int(len(df_sorted) * 0.85)
train_df = df_sorted.iloc[:train_end]
calibration_df = df_sorted.iloc[train_end:cal_end]
test_df = df_sorted.iloc[cal_end:]

baseline_predictions = test_df['prior_leg_delay'].to_numpy()
baseline_mae = float(np.mean(np.abs(test_df['actual_delay_minutes'].to_numpy() - baseline_predictions)))
print(f"Chronological held-out baseline MAE: {baseline_mae:.2f} minutes")

## 3. Evaluate XGBoost (Chronological Held-Out)

XGBoost is **fit only on the training split** (first 70% chronologically) and evaluated on the **same held-out test slice** as the baseline above. No future data is used during training.


In [ ]:
model = build_model()
model.fit(train_df[FEATURES], train_df['actual_delay_minutes'])
test_predictions = model.predict(test_df[FEATURES])
xgb_mae = float(np.mean(np.abs(test_df['actual_delay_minutes'].to_numpy() - test_predictions)))
print(f"Chronological held-out XGBoost MAE: {xgb_mae:.2f} minutes")
print(f"Held-out rows: {len(test_df)}")

## 4. Summary & Impact

In [ ]:
improvement = ((baseline_mae - xgb_mae) / baseline_mae) * 100
print("=== HELD-OUT IMPACT SUMMARY ===")
print(f"Baseline Error : {baseline_mae:.1f} minutes")
print(f"XGBoost Error  : {xgb_mae:.1f} minutes")
print(f"Improvement    : {improvement:.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['Naive Baseline', 'XGBoost'], [baseline_mae, xgb_mae], color=['gray', 'blue'])
ax.set_ylabel('Mean Absolute Error (Minutes)')
ax.set_title('Chronological Held-Out ETA Prediction Error')
for i, value in enumerate([baseline_mae, xgb_mae]):
    ax.text(i, value - 2, f"{value:.1f} min", ha='center', color='white', fontweight='bold')
plt.show()

## 5. Empirical Conformal Coverage

The chronological 70/15/15 split reserves the final 15% as untouched test data. The reported percentage is measured on those held-out rows, not assumed from the nominal MAPIE target.

In [ ]:
import sys
sys.path.append('..')
from src.calibration.conformal import train_and_calibrate

calibrated_engine, calibration_metrics = train_and_calibrate(df)
print('=== HELD-OUT CONFORMAL VALIDATION ===')
for name, value in calibration_metrics.items():
    print(f'{name}: {value}')

print(f"Measured P10-P90 coverage: {calibration_metrics['coverage_90_pct']:.1f}%")
print('Target coverage: 90.0%')